# Setup
Notebooks call reusable functions from `src/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve_paths, set_global_seed, get_seed
config = load_config(ROOT / 'configs/project_config.yaml')
paths = resolve_paths(config)
set_global_seed(get_seed(config))
print('project root:', paths.root)


## Ablation and horizon robustness

In [ ]:
import joblib
from src.data_loader import load_parquet
from src.temporal_split import chronological_date_split, masks_from_split
from src.ablation import run_ablation
from src.robustness import run_horizon_robustness
df = load_parquet(paths.processed).sort_values('timestamp').reset_index(drop=True)
split = chronological_date_split(df, 0.6, 0.2, 0.2, 60)
masks = masks_from_split(len(df), split)
valid = df['label'].notna().to_numpy()
for k in masks: masks[k] &= valid
bundle = joblib.load(paths.models/'xgboost_model.joblib')
abl = run_ablation(df, masks, config, best_params=bundle.get('best_params'), tables_dir=paths.tables)
hor = run_horizon_robustness(df, masks, config, best_params=bundle.get('best_params'), tables_dir=paths.tables)
print(abl)
print(hor)
